<!-- CELL 1 — MARKDOWN -->
# Phase 2 — Preprocessing
### Uplift Modeling on the Hillstrom Email Marketing Dataset

**Project:** `uplift-causal-ml` — a 7-phase causal machine learning project on uplift modeling.

**Recap of Phase 1:** the Hillstrom dataset (64,000 rows, 0 missing values) was loaded and
validated, with a binary `treatment` column (1 = received email, 0 = control) derived from
the raw 3-arm `segment` column, and the randomization check passed (no covariates were
imbalanced between treatment and control).

**This notebook (Phase 2) covers:**
1. Loading the same dataset the same way Phase 1 did
2. One-hot encoding the categorical features
3. Building the final feature matrix `X`, keeping treatment and outcomes separate
4. An 80/20 train/test split, stratified on treatment **and** outcome jointly
5. Verifying the split preserved the treatment/outcome rates from Phase 1
6. Saving the processed splits to `data/processed/`
7. A closing summary of what Phase 2 produced


<!-- CELL 2 — MARKDOWN -->
## 1. Load the dataset

In [1]:
# ---------------------------------------------------------------------------
# CELL 3 — CODE: Load the dataset the same way Phase 1 did, via the shared
# `load_hillstrom()` loader in src/data_loader.py (sklift download, with a
# local-cache fallback), then re-derive the binary `treatment` column exactly
# as Phase 1 did (1 = received any email, 0 = control / no email).
# ---------------------------------------------------------------------------
import sys
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

sys.path.append(os.path.abspath(os.path.join("..")))
from src.data_loader import load_hillstrom

pd.set_option("display.max_columns", None)

df = load_hillstrom()

treatment_col = "treatment"
raw_treatment_col = "segment"
outcome_cols = ["visit", "conversion", "spend"]
categorical_features = ["history_segment", "zip_code", "channel"]
numeric_features = ["recency", "history", "mens", "womens", "newbie"]
feature_cols = numeric_features + categorical_features

# Binarize treatment exactly as in Phase 1: any email campaign vs. no email
df["treatment"] = (df["segment"] != "No E-Mail").astype(int)

print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Loaded dataset via sklift.datasets.fetch_hillstrom().
Loaded dataset: 64,000 rows x 13 columns


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,treatment
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0,1
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0,0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0,1
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0,1
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0,1


<!-- CELL 4 — MARKDOWN -->
## 2. One-hot encode categorical features

`history_segment`, `zip_code`, and `channel` are nominal categories (no natural order), so
we one-hot encode them with `pd.get_dummies`. The five numeric features
(`recency`, `history`, `mens`, `womens`, `newbie`) are left as-is -- `mens`/`womens`/`newbie`
are already binary 0/1 flags, and `recency`/`history` are continuous.

We use `drop_first=False` (keep all dummy columns) since this is EDA/preprocessing for
tree-based and general uplift models later, where the redundant column from
multicollinearity isn't a practical issue and the full one-hot representation is easier
to interpret in a report.


In [2]:
# ---------------------------------------------------------------------------
# CELL 5 — CODE: One-hot encode the three categorical features. Numeric
# features are left untouched. `dtype=int` keeps the dummy columns as 0/1
# integers rather than booleans, for consistency with the rest of the data.
# ---------------------------------------------------------------------------
encoded_categoricals = pd.get_dummies(
    df[categorical_features], columns=categorical_features, drop_first=False, dtype=int
)

print(f"Categorical columns before encoding : {categorical_features}")
print(f"Columns after one-hot encoding       : {encoded_categoricals.shape[1]}")
print()
encoded_categoricals.head()


Categorical columns before encoding : ['history_segment', 'zip_code', 'channel']
Columns after one-hot encoding       : 13



,history_segment_1) $0 - $100,history_segment_2) $100 - $200,history_segment_3) $200 - $350,history_segment_4) $350 - $500,history_segment_5) $500 - $750,"history_segment_6) $750 - $1,000","history_segment_7) $1,000 +",zip_code_Rural,zip_code_Surburban,zip_code_Urban,channel_Multichannel,channel_Phone,channel_Web
0,0,1,0,0,0,0,0,0,1,0,0,1,0
1,0,0,1,0,0,0,0,1,0,0,0,0,1
2,0,1,0,0,0,0,0,0,1,0,0,0,1
3,0,0,0,0,1,0,0,1,0,0,0,0,1
4,1,0,0,0,0,0,0,0,0,1,0,0,1


<!-- CELL 6 — MARKDOWN -->
## 3. Build the final feature matrix X

In [3]:
# ---------------------------------------------------------------------------
# CELL 7 — CODE: Assemble the final feature matrix X (numeric features +
# one-hot encoded categoricals). Treatment and the three outcome columns
# are kept as SEPARATE objects -- they must never leak into X, since X is
# what later-phase models will see as "features".
# ---------------------------------------------------------------------------
X = pd.concat([df[numeric_features], encoded_categoricals], axis=1)
treatment = df[treatment_col].copy()
y = df[outcome_cols].copy()

print(f"X shape          : {X.shape}")
print(f"treatment shape  : {treatment.shape}")
print(f"y shape          : {y.shape}  (columns: {y.columns.tolist()})")
print()
print("X columns:")
print(X.columns.tolist())

assert treatment_col not in X.columns, "Treatment column leaked into X!"
assert not any(c in X.columns for c in outcome_cols), "An outcome column leaked into X!"


X shape          : (64000, 18)
treatment shape  : (64000,)
y shape          : (64000, 3)  (columns: ['visit', 'conversion', 'spend'])

X columns:
['recency', 'history', 'mens', 'womens', 'newbie', 'history_segment_1) $0 - $100', 'history_segment_2) $100 - $200', 'history_segment_3) $200 - $350', 'history_segment_4) $350 - $500', 'history_segment_5) $500 - $750', 'history_segment_6) $750 - $1,000', 'history_segment_7) $1,000 +', 'zip_code_Rural', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Multichannel', 'channel_Phone', 'channel_Web']


<!-- CELL 8 — MARKDOWN -->
## 4. Train/test split (80/20), stratified on treatment + outcome jointly

A plain stratified split on `treatment` alone would keep the treatment/control ratio
consistent between train and test, but conversion is rare (~0.6-1.1%) -- a plain split
could still let the *conversion rate within each arm* drift between train and test by
chance. To avoid that, we stratify on a **combined key**: `treatment` and `conversion`
concatenated together, so all four combinations below are proportionally represented in
both train and test:

- treated + converted
- treated + not converted
- control + converted
- control + not converted

We use `conversion` (not `visit` or `spend`) as the outcome half of the stratification key,
since it's the rarest/most imbalanced of the three outcomes and therefore the one most at
risk of drifting in a plain random split.


In [4]:
# ---------------------------------------------------------------------------
# CELL 9 — CODE: Build the combined treatment+conversion stratification key,
# then perform an 80/20 train/test split stratified on that key.
# ---------------------------------------------------------------------------
strat_key = treatment.astype(str) + "_" + y["conversion"].astype(str)

print("Stratification key value counts (treatment_conversion):")
print(strat_key.value_counts())
print()

X_train, X_test, treatment_train, treatment_test, y_train, y_test = train_test_split(
    X, treatment, y,
    test_size=0.20,
    random_state=42,
    stratify=strat_key,
)

print(f"\nTrain set: {X_train.shape[0]:,} rows")
print(f"Test set : {X_test.shape[0]:,} rows")


Stratification key value counts (treatment_conversion):
1_0    42238
0_0    21184
1_1      456
0_1      122
Name: count, dtype: int64


Train set: 51,200 rows
Test set : 12,800 rows


<!-- CELL 10 — MARKDOWN -->
## 5. Verify the split preserved Phase 1's rates

In [5]:
# ---------------------------------------------------------------------------
# CELL 11 — CODE: For each split (train, test) and each outcome (visit,
# conversion), compute the treatment/control rate and compare it against
# the full-dataset numbers from Phase 1. Flag any split/outcome/group
# combination that deviates by more than a small tolerance.
# ---------------------------------------------------------------------------
# Phase 1 baseline reference numbers (full dataset)
phase1_reference = {
    ("conversion", 0): 0.0057,  # control conversion rate
    ("conversion", 1): 0.0107,  # treatment conversion rate
    ("visit", 0): 0.1062,       # control visit rate
    ("visit", 1): 0.1670,       # treatment visit rate
}

# Phase 1 full-dataset treatment/control split, as percentages
REFERENCE_CONTROL_PCT = 33.29
REFERENCE_TREATMENT_PCT = 66.71

# Tolerance for flagging a deviation as noticeable (absolute rate/percentage difference)
TOLERANCE = 0.01  # 1 percentage point

def treatment_ratio(treat_series):
    n = len(treat_series)
    n_treat = int(treat_series.sum())
    return n_treat, n - n_treat, n_treat / n * 100, (n - n_treat) / n * 100

def outcome_rate_by_group(y_series, treat_series, outcome):
    rates = y_series.groupby(treat_series)[outcome].mean()
    return rates.get(0, np.nan), rates.get(1, np.nan)

verification_rows = []
for split_name, treat_series, y_series in [
    ("train", treatment_train, y_train),
    ("test", treatment_test, y_test),
]:
    n_treat, n_control, pct_treat, pct_control = treatment_ratio(treat_series)

    # Compare this split's treatment/control percentages against the Phase 1
    # full-dataset split (33.29% control / 66.71% treatment). Guaranteed to be
    # "ok" by construction (the stratified split preserves this ratio exactly),
    # but we check and flag it explicitly for consistency with the outcome rows.
    control_pct_dev = abs(pct_control - REFERENCE_CONTROL_PCT)
    treat_pct_dev = abs(pct_treat - REFERENCE_TREATMENT_PCT)
    split_flag = "DEVIATES" if (control_pct_dev > TOLERANCE * 100 or treat_pct_dev > TOLERANCE * 100) else "ok"

    verification_rows.append({
        "split": split_name, "metric": "treatment/control split",
        "control": f"{n_control:,} ({pct_control:.2f}%, ref {REFERENCE_CONTROL_PCT:.2f}%)",
        "treatment": f"{n_treat:,} ({pct_treat:.2f}%, ref {REFERENCE_TREATMENT_PCT:.2f}%)",
        "flag": split_flag,
    })

    for outcome in ["visit", "conversion"]:
        control_rate, treat_rate = outcome_rate_by_group(y_series, treat_series, outcome)

        ref_control = phase1_reference[(outcome, 0)]
        ref_treat = phase1_reference[(outcome, 1)]
        control_dev = abs(control_rate - ref_control)
        treat_dev = abs(treat_rate - ref_treat)
        flag = "DEVIATES" if (control_dev > TOLERANCE or treat_dev > TOLERANCE) else "ok"

        verification_rows.append({
            "split": split_name, "metric": f"{outcome} rate",
            "control": f"{control_rate*100:.2f}% (ref {ref_control*100:.2f}%)",
            "treatment": f"{treat_rate*100:.2f}% (ref {ref_treat*100:.2f}%)",
            "flag": flag,
        })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

any_deviation = (verification_df["flag"] == "DEVIATES").any()
print()
if any_deviation:
    print("FLAGGED: one or more train/test rates deviate noticeably from the Phase 1 "
          f"full-dataset numbers (tolerance = {TOLERANCE*100:.0f} pp). See rows above.")
else:
    print(f"All train/test rates are within {TOLERANCE*100:.0f} percentage point of the "
          "Phase 1 full-dataset numbers -- the split preserved treatment/outcome balance.")


,split,metric,control,treatment,flag
0,train,treatment/control split,"17,045 (33.29%, ref 33.29%)","34,155 (66.71%, ref 66.71%)",ok
1,train,visit rate,10.51% (ref 10.62%),16.73% (ref 16.70%),ok
2,train,conversion rate,0.57% (ref 0.57%),1.07% (ref 1.07%),ok
3,test,treatment/control split,"4,261 (33.29%, ref 33.29%)","8,539 (66.71%, ref 66.71%)",ok
4,test,visit rate,11.03% (ref 10.62%),16.59% (ref 16.70%),ok
5,test,conversion rate,0.56% (ref 0.57%),1.07% (ref 1.07%),ok



All train/test rates are within 1 percentage point of the Phase 1 full-dataset numbers -- the split preserved treatment/outcome balance.


<!-- CELL 12 — MARKDOWN -->
## 6. Save the processed splits to `data/processed/`

In [6]:
# ---------------------------------------------------------------------------
# CELL 13 — CODE: Save X_train/X_test, treatment_train/treatment_test, and
# y_train/y_test (all 3 outcome columns) to data/processed/ as CSVs, so
# later phases can load them directly without repeating this preprocessing.
# ---------------------------------------------------------------------------
processed_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

X_train.to_csv(os.path.join(processed_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(processed_dir, "X_test.csv"), index=False)
treatment_train.to_csv(os.path.join(processed_dir, "treatment_train.csv"), index=False, header=["treatment"])
treatment_test.to_csv(os.path.join(processed_dir, "treatment_test.csv"), index=False, header=["treatment"])
y_train.to_csv(os.path.join(processed_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(processed_dir, "y_test.csv"), index=False)

saved_files = [
    "X_train.csv", "X_test.csv",
    "treatment_train.csv", "treatment_test.csv",
    "y_train.csv", "y_test.csv",
]
print("Saved processed splits to data/processed/:")
for fname in saved_files:
    fpath = os.path.join(processed_dir, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<24} ({size_kb:,.1f} KB)")


Saved processed splits to data/processed/:
  X_train.csv              (2,089.4 KB)
  X_test.csv               (522.8 KB)
  treatment_train.csv      (150.0 KB)
  treatment_test.csv       (37.5 KB)
  y_train.csv              (451.1 KB)
  y_test.csv               (112.8 KB)


<!-- CELL 14 — MARKDOWN -->
## 7. Phase 2 summary

In [7]:
# ---------------------------------------------------------------------------
# CELL 15 — CODE: Pull together the final feature count, train/test sizes,
# and the post-split balance verdict from Cell 11 into one closing summary
# -- the deliverable for Phase 2.
# ---------------------------------------------------------------------------
n_features_final = X.shape[1]
n_numeric = len(numeric_features)
n_categorical_encoded = encoded_categoricals.shape[1]

summary_md = f"""
### Phase 2 Summary — Preprocessing

- **Final feature count:** {n_features_final} features
  ({n_numeric} numeric + {n_categorical_encoded} one-hot encoded from
  {len(categorical_features)} categorical columns: {categorical_features}).
- **Train set:** {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.0f}%)
- **Test set:** {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.0f}%)
- **Stratification:** split on a combined treatment + conversion key, so all four
  (treated/control x converted/not-converted) combinations are proportionally represented
  in both sets.
- **Post-split balance check:** {'**FLAGGED** -- see the verification table above for which rate(s) deviated.' if any_deviation else '**PASSED** -- train and test treatment/control ratios and outcome rates all matched the Phase 1 full-dataset numbers within tolerance.'}
- **Saved outputs:** `data/processed/` now contains `X_train.csv`, `X_test.csv`,
  `treatment_train.csv`, `treatment_test.csv`, `y_train.csv`, `y_test.csv`
  (y files include all three outcome columns: visit, conversion, spend).

**Next steps (Phase 3+):** use these saved splits to train uplift models (e.g. two-model
approach, class transformation, or uplift trees) on the training set and evaluate on the
held-out test set.
"""

display(Markdown(summary_md))



### Phase 2 Summary — Preprocessing

- **Final feature count:** 18 features
  (5 numeric + 13 one-hot encoded from
  3 categorical columns: ['history_segment', 'zip_code', 'channel']).
- **Train set:** 51,200 rows (80%)
- **Test set:** 12,800 rows (20%)
- **Stratification:** split on a combined treatment + conversion key, so all four
  (treated/control x converted/not-converted) combinations are proportionally represented
  in both sets.
- **Post-split balance check:** **PASSED** -- train and test treatment/control ratios and outcome rates all matched the Phase 1 full-dataset numbers within tolerance.
- **Saved outputs:** `data/processed/` now contains `X_train.csv`, `X_test.csv`,
  `treatment_train.csv`, `treatment_test.csv`, `y_train.csv`, `y_test.csv`
  (y files include all three outcome columns: visit, conversion, spend).

**Next steps (Phase 3+):** use these saved splits to train uplift models (e.g. two-model
approach, class transformation, or uplift trees) on the training set and evaluate on the
held-out test set.
